In [1]:
# ============================================================
# UNDERWATER OBJECT DETECTION PIPELINE
# WoS-Standard: CBAM + Hybrid Loss + Multi-Dataset DG
# ============================================================

# ============================================================
# CELL 1: Install Dependencies
# ============================================================
!pip install ultralytics pytorch-msssim -q

# ============================================================
# CELL 2: Imports and Configuration
# ============================================================
import os
import shutil
import glob
import yaml
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms, models
from PIL import Image
import numpy as np
from pytorch_msssim import ssim
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as pe
import random

# ─── Paths: Enhancement Datasets ────────────────────────────
UIEB_RAW = "/kaggle/input/datasets/larjeck/uieb-dataset-raw/raw-890"
UIEB_REF = "/kaggle/input/datasets/larjeck/uieb-dataset-reference/reference-890"

# LSUI – Large Scale Underwater Image dataset
# Expected layout: /kaggle/input/.../lsui/input/   (raw)
#                  /kaggle/input/.../lsui/GT/       (reference)
LSUI_RAW = "/kaggle/input/datasets/noureldin199/lsui-large-scale-underwater-image-dataset/LSUI/input"
LSUI_REF = "/kaggle/input/datasets/noureldin199/lsui-large-scale-underwater-image-dataset/LSUI/GT"

# EUVP – Enhancing Underwater Visual Perception
# Expected layout: /kaggle/input/.../euvp/trainA/  (raw)
#                  /kaggle/input/.../euvp/trainB/  (reference)
EUVP_RAW = "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagenet/trainA"
EUVP_REF = "/kaggle/input/datasets/pamuduranasinghe/euvp-dataset/EUVP/Paired/underwater_imagenet/trainB"

# ─── Paths: Detection Datasets ──────────────────────────────
RUOD_TRAIN_IMG = "/kaggle/input/datasets/nikitha1317/ruod-dataset1/train/images"
RUOD_TRAIN_LBL = "/kaggle/input/datasets/nikitha1317/ruod-dataset1/train/labels"
RUOD_VAL_IMG   = "/kaggle/input/datasets/nikitha1317/ruod-dataset1/valid/images"
RUOD_VAL_LBL   = "/kaggle/input/datasets/nikitha1317/ruod-dataset1/valid/labels"
RUOD_TEST_IMG  = "/kaggle/input/datasets/nikitha1317/ruod-dataset1/test/images"
RUOD_TEST_LBL  = "/kaggle/input/datasets/nikitha1317/ruod-dataset1/test/labels"

# LSUI / EUVP detection splits (if labelled subsets exist)
# Set these to None to skip that domain during YOLO evaluation
LSUI_TEST_IMG  = "/kaggle/input/lsui-dataset/test/images"
LSUI_TEST_LBL  = "/kaggle/input/lsui-dataset/test/labels"
EUVP_TEST_IMG  = "/kaggle/input/euvp-dataset/test/images"
EUVP_TEST_LBL  = "/kaggle/input/euvp-dataset/test/labels"

WORKING      = "/kaggle/working"
ENHANCER_PATH = f"{WORKING}/enhancer_cbam.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ============================================================
# CELL 3: Paired Image Dataset (generic – works for all sets)
# ============================================================
class PairedImageDataset(Dataset):
    """
    Generic paired dataset: raw (degraded) → reference (clean).
    Skips pairs where the reference file is missing.
    Supports jpg, jpeg, png, bmp, tif.
    """
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

    def __init__(self, raw_dir, ref_dir, size=256, is_train=True):
        self.raw_files = sorted(glob.glob(os.path.join(raw_dir, "*.*"))) # Simplified for brevity
        self.ref_dir = ref_dir
        self.is_train = is_train
        
        # BASIC TRANSFORMS
        self.base_transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
        ])
        
        # CHAOS AUGMENTATION (Part 2 Requirement)
        self.chaos_transform = transforms.Compose([
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        ])
    def __len__(self):
        return len(self.raw_files)
    def apply_physics_ifm(self, img_tensor):
        """Reviewer-Grade Physics: Wavelength Decay + Spatial Depth Map"""
        # Underwater coefficients: Red fades fast (0.5), Green medium (0.2), Blue slow (0.1)
        beta_r, beta_g, beta_b = 0.5, 0.2, 0.1
        
        # Create a spatial depth map (0.5m to 2.0m) to simulate non-uniform water
        # Higher values = deeper water = more attenuation
        depth_map = torch.rand((1, img_tensor.shape[1], img_tensor.shape[2]), device=img_tensor.device) * 1.5 + 0.5
        
        # Apply exponential decay: I = I_0 * exp(-beta * depth)
        img_tensor[0] *= torch.exp(-beta_r * depth_map).squeeze(0) # Red
        img_tensor[1] *= torch.exp(-beta_g * depth_map).squeeze(0) # Green
        img_tensor[2] *= torch.exp(-beta_b * depth_map).squeeze(0) # Blue
        
        return torch.clamp(img_tensor, 0, 1)

    def __getitem__(self, idx):
        raw = Image.open(self.raw_files[idx]).convert("RGB")
        ref = Image.open(os.path.join(self.ref_dir, os.path.basename(self.raw_files[idx]))).convert("RGB")
        
        raw_t = self.base_transform(raw)
        ref_t = self.base_transform(ref)
        
        if self.is_train:
            if self.is_train:
                raw = self.chaos_transform(raw)   # apply on PIL
            raw_t = self.base_transform(raw)# Apply Chaos
            raw_t = self.apply_physics_ifm(raw_t) # Apply Physics-IFM
            
        return raw_t, ref_t
            
def apply_underwater_ifm(tensor):
    # Ensure tint_color is on the same device as input tensor
    tint_color = torch.tensor([0.4, 0.9, 0.8] if random.random() > 0.5 else [0.4, 0.7, 0.9], device=tensor.device)
    tint_color = tint_color.view(3, 1, 1)
    
    factor = random.uniform(0.1, 0.3)
    tensor = (1 - factor) * tensor + factor * tint_color
    
    mean = tensor.mean()
    tensor = (tensor - mean) * random.uniform(0.7, 0.9) + mean
    return tensor.clamp(0, 1)

def build_enhancement_loader(batch_size: int = 16, num_workers: int = 2):
    """
    Combines UIEB + LSUI + EUVP into a single DataLoader.
    Datasets whose directories do not exist are silently skipped.
    """
    datasets = []
    configs  = [
        ("UIEB",  UIEB_RAW,  UIEB_REF),
        ("LSUI",  LSUI_RAW,  LSUI_REF),
        ("EUVP",  EUVP_RAW,  EUVP_REF),
    ]
    for name, raw_dir, ref_dir in configs:
        if os.path.isdir(raw_dir) and os.path.isdir(ref_dir):
            ds = PairedImageDataset(raw_dir, ref_dir)
            print(f"  [{name}] {len(ds):>5} pairs found")
            datasets.append(ds)
        else:
            print(f"  [{name}] SKIPPED – path not found ({raw_dir})")

    if not datasets:
        raise RuntimeError("No enhancement datasets found. Check your input paths.")

    combined = ConcatDataset(datasets)
    loader   = DataLoader(combined, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
    print(f"  Combined: {len(combined):>5} pairs | batches: {len(loader)}")
    return loader


print("Building enhancement DataLoader …")
uieb_loader = build_enhancement_loader()

# ============================================================
# CELL 4: CBAM – Convolutional Block Attention Module
# ============================================================
class ChannelAttention(nn.Module):
    """Squeeze-and-Excitation style channel attention."""
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.shared_mlp = nn.Sequential(
            nn.Conv2d(channels, mid, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid, channels, 1, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_w = self.shared_mlp(self.avg_pool(x))
        max_w = self.shared_mlp(self.max_pool(x))
        return self.sigmoid(avg_w + max_w)


class SpatialAttention(nn.Module):
    """Spatial attention using avg+max channel descriptors."""
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_map = x.mean(dim=1, keepdim=True)
        max_map = x.max(dim=1, keepdim=True).values
        desc    = torch.cat([avg_map, max_map], dim=1)
        return self.sigmoid(self.conv(desc))


class CBAM(nn.Module):
    """Full CBAM: channel attention → spatial attention."""
    def __init__(self, channels: int, reduction: int = 16, spatial_k: int = 7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_k)

    def forward(self, x):
        x = x * self.ca(x)
        x = x * self.sa(x)
        return x

# ============================================================
# CELL 5: Enhanced ConvBlock with Residual + CBAM
# ============================================================
class ConvBlock(nn.Module):
    """
    Double-conv block with:
      • Residual (skip) connection  – helps gradient flow
      • CBAM attention              – focuses on salient features
    """
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.cbam     = CBAM(out_ch)
        # 1×1 projection for residual when channel dims differ
        self.residual = (
            nn.Conv2d(in_ch, out_ch, 1, bias=False)
            if in_ch != out_ch else nn.Identity()
        )

    def forward(self, x):
        out = self.block(x)
        out = self.cbam(out)
        return out + self.residual(x)   # residual add

# ============================================================
# CELL 6: LightUNet with CBAM + Residual Connections
# ============================================================
class LightUNet(nn.Module):
    """
    Lightweight U-Net enhanced with:
      • CBAM attention in every ConvBlock
      • Residual skip connections within each ConvBlock
      • U-Net long-range skip connections (encoder → decoder)
    """
    def __init__(self):
        super().__init__()
        # ── Encoder ──────────────────────────────────────────
        self.enc1 = ConvBlock(3,   16)
        self.enc2 = ConvBlock(16,  32)
        self.enc3 = ConvBlock(32,  64)
        self.pool = nn.MaxPool2d(2)

        # ── Bottleneck ───────────────────────────────────────
        self.bottleneck = ConvBlock(64, 128)

        # ── Decoder ──────────────────────────────────────────
        self.up3  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = ConvBlock(128, 64)   # 64 up + 64 skip

        self.up2  = nn.ConvTranspose2d(64,  32, 2, stride=2)
        self.dec2 = ConvBlock(64,  32)   # 32 up + 32 skip

        self.up1  = nn.ConvTranspose2d(32,  16, 2, stride=2)
        self.dec1 = ConvBlock(32,  16)   # 16 up + 16 skip

        self.out     = nn.Conv2d(16, 3, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))

        d3 = self.dec3(torch.cat([self.up3(b),  e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.sigmoid(self.out(d1))


model = LightUNet().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Enhancer parameters: {total_params:,}")

# ============================================================
# CELL 7: Hybrid Loss (L1 + SSIM + Perceptual + Color Constancy)
# ============================================================
class PerceptualLoss(nn.Module):
    """
    Uses VGG-16 relu2_2 and relu3_3 feature maps.
    Frozen backbone – only features are used, not trained.
    Memory-efficient: extracts only needed layers.
    """
    VGG_MEAN = [0.485, 0.456, 0.406]
    VGG_STD  = [0.229, 0.224, 0.225]

    def __init__(self, device):
        super().__init__()
        vgg      = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        # relu2_2 = index 9  |  relu3_3 = index 16
        self.slice1 = nn.Sequential(*list(vgg.children())[:10]).to(device)
        self.slice2 = nn.Sequential(*list(vgg.children())[10:17]).to(device)
        for p in self.parameters():
            p.requires_grad_(False)

        mean = torch.tensor(self.VGG_MEAN, device=device).view(1, 3, 1, 1)
        std  = torch.tensor(self.VGG_STD,  device=device).view(1, 3, 1, 1)
        self.register_buffer("mean", mean)
        self.register_buffer("std",  std)
        self.criterion = nn.L1Loss()

    def _normalize(self, x):
        return (x - self.mean) / self.std

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred_n, tgt_n = self._normalize(pred), self._normalize(target)
        # relu2_2
        f_pred1  = self.slice1(pred_n)
        f_tgt1   = self.slice1(tgt_n)
        loss     = self.criterion(f_pred1, f_tgt1)
        # relu3_3
        f_pred2  = self.slice2(f_pred1)
        f_tgt2   = self.slice2(f_tgt1)
        loss    += self.criterion(f_pred2, f_tgt2)
        return loss


def color_constancy_loss(pred: torch.Tensor) -> torch.Tensor:
    """
    Penalises colour cast (green/blue dominance in underwater).
    For each pair of channels, the mean over the spatial dims
    should be equal (grey-world assumption).
    Lower = more balanced colours.
    """
    mean_r = pred[:, 0, :, :].mean(dim=[1, 2])
    mean_g = pred[:, 1, :, :].mean(dim=[1, 2])
    mean_b = pred[:, 2, :, :].mean(dim=[1, 2])
    loss   = (
        (mean_r - mean_g).pow(2) +
        (mean_r - mean_b).pow(2) +
        (mean_g - mean_b).pow(2)
    ).mean()
    return loss


class HybridEnhancementLoss(nn.Module):
    """
    Weighted combination of:
      λ1 · L1  +  λ2 · (1 – SSIM)  +  λ3 · Perceptual  +  λ4 · ColorConstancy
    Defaults tuned for underwater imagery.
    """
    def __init__(
        self,
        device,
        lambda_l1:    float = 1.0,
        lambda_ssim:  float = 0.5,
        lambda_perc:  float = 0.1,
        lambda_color: float = 0.05,
    ):
        super().__init__()
        self.l1         = nn.L1Loss()
        self.perceptual = PerceptualLoss(device)
        self.lam_l1     = lambda_l1
        self.lam_ssim   = lambda_ssim
        self.lam_perc   = lambda_perc
        self.lam_color  = lambda_color

    def forward(
        self, pred: torch.Tensor, target: torch.Tensor
    ) -> tuple[torch.Tensor, dict]:
        l_l1    = self.l1(pred, target)
        l_ssim  = 1.0 - ssim(pred, target, data_range=1.0, size_average=True)
        l_perc  = self.perceptual(pred, target)
        l_color = color_constancy_loss(pred)

        total = (
            self.lam_l1   * l_l1   +
            self.lam_ssim * l_ssim +
            self.lam_perc * l_perc +
            self.lam_color * l_color
        )
        breakdown = {
            "l1": l_l1.item(),
            "ssim": l_ssim.item(),
            "perc": l_perc.item(),
            "color": l_color.item(),
        }
        return total, breakdown

# ============================================================
# CELL 8: Train Enhancement Model
# ============================================================
criterion = HybridEnhancementLoss(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)

EPOCHS = 10

print("\n" + "="*60)
print(" Training LightUNet-CBAM with Hybrid Loss")
print("="*60)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    breakdown_accum = {"l1": 0, "ssim": 0, "perc": 0, "color": 0}

    for raw, ref in uieb_loader:
        raw, ref = raw.to(DEVICE), ref.to(DEVICE)
        if random.random() > 0.7: raw = apply_underwater_ifm(raw)
        optimizer.zero_grad()

        pred = model(raw)
        loss, bd = criterion(pred, ref)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        for k in breakdown_accum:
            breakdown_accum[k] += bd[k]

    scheduler.step()
    n = len(uieb_loader)
    print(
        f"Epoch [{epoch:>2}/{EPOCHS}] "
        f"Loss: {total_loss/n:.4f} | "
        f"L1: {breakdown_accum['l1']/n:.4f} | "
        f"SSIM: {breakdown_accum['ssim']/n:.4f} | "
        f"Perc: {breakdown_accum['perc']/n:.4f} | "
        f"Color: {breakdown_accum['color']/n:.4f} | "
        f"LR: {scheduler.get_last_lr()[0]:.2e}"
    )

torch.save(model.state_dict(), ENHANCER_PATH)
print(f"\nEnhancer saved → {ENHANCER_PATH}")

# ============================================================
# CELL 9: Enhancement Inference Function
# ============================================================
def enhance_images(
    enhancer: nn.Module,
    src_img_dir: str,
    dst_img_dir: str,
    device,
    batch_size: int = 32,
    img_size:   int = 256,
):
    """
    Enhance all images in src_img_dir and write results to dst_img_dir.
    Restores original spatial resolution after enhancement.
    """
    os.makedirs(dst_img_dir, exist_ok=True)

    transform = transforms.ToTensor()
    to_pil    = transforms.ToPILImage()
    resize_in = transforms.Resize((img_size, img_size))

    img_paths = sorted(
        glob.glob(os.path.join(src_img_dir, "*.jpg"))  +
        glob.glob(os.path.join(src_img_dir, "*.jpeg")) +
        glob.glob(os.path.join(src_img_dir, "*.png"))
    )
    if not img_paths:
        print(f"  [SKIP] No images found in {src_img_dir}")
        return

    enhancer.eval()
    with torch.no_grad():
        for i in range(0, len(img_paths), batch_size):
            batch_paths  = img_paths[i:i + batch_size]
            originals    = [Image.open(p).convert("RGB") for p in batch_paths]
            orig_sizes   = [img.size for img in originals]   # (W, H)

            tensors = torch.stack([
                resize_in(transform(img)) for img in originals
            ]).to(device)

            outputs = enhancer(tensors).cpu()

            for out_t, orig_size, path in zip(outputs, orig_sizes, batch_paths):
                fname   = os.path.basename(path)
                out_img = to_pil(out_t.clamp(0, 1))
                out_img = out_img.resize(orig_size, Image.BILINEAR)
                out_img.save(os.path.join(dst_img_dir, fname))

    print(f"  Enhanced {len(img_paths)} images → {dst_img_dir}")


# Load (or reuse) trained enhancer
enhancer = LightUNet().to(DEVICE)
enhancer.load_state_dict(torch.load(ENHANCER_PATH, map_location=DEVICE))

# ============================================================
# CELL 10: Enhance RUOD Images (Train / Val / Test)
# ============================================================
ENH_RUOD_TRAIN = f"{WORKING}/enhanced/ruod/train/images"
ENH_RUOD_VAL   = f"{WORKING}/enhanced/ruod/valid/images"
ENH_RUOD_TEST  = f"{WORKING}/enhanced/ruod/test/images"

print("\nEnhancing RUOD …")
enhance_images(enhancer, RUOD_TRAIN_IMG, ENH_RUOD_TRAIN, DEVICE)
enhance_images(enhancer, RUOD_VAL_IMG,   ENH_RUOD_VAL,   DEVICE)
enhance_images(enhancer, RUOD_TEST_IMG,  ENH_RUOD_TEST,  DEVICE)

# Enhance LSUI / EUVP test sets (only if they exist)
ENH_LSUI_TEST  = f"{WORKING}/enhanced/lsui/test/images"
ENH_EUVP_TEST  = f"{WORKING}/enhanced/euvp/test/images"

if os.path.isdir(LSUI_TEST_IMG):
    print("Enhancing LSUI test …")
    enhance_images(enhancer, LSUI_TEST_IMG, ENH_LSUI_TEST, DEVICE)

if os.path.isdir(EUVP_TEST_IMG):
    print("Enhancing EUVP test …")
    enhance_images(enhancer, EUVP_TEST_IMG, ENH_EUVP_TEST, DEVICE)

# ============================================================
# CELL 11: Build UDA (Unified Domain Augmented) Dataset
# ============================================================
def build_uda_split(
    orig_img_dir: str,
    orig_lbl_dir: str,
    enh_img_dir:  str,
    out_img_dir:  str,
    out_lbl_dir:  str,
    prefix:       str = "enh_",
):
    """
    Merges original + enhanced images into one detection split.
    Labels are shared (YOLO normalised coords are enhancement-invariant).
    """
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    # Original images + labels
    for img_path in glob.glob(os.path.join(orig_img_dir, "*")):
        shutil.copy(img_path, out_img_dir)

    for lbl_path in glob.glob(os.path.join(orig_lbl_dir, "*.txt")):
        shutil.copy(lbl_path, out_lbl_dir)

    # Enhanced images + duplicated labels
    for img_path in glob.glob(os.path.join(enh_img_dir, "*")):
        fname    = os.path.basename(img_path)
        stem     = Path(fname).stem
        new_name = f"{prefix}{fname}"
        shutil.copy(img_path, os.path.join(out_img_dir, new_name))

        lbl_src = os.path.join(orig_lbl_dir, f"{stem}.txt")
        if os.path.exists(lbl_src):
            shutil.copy(lbl_src, os.path.join(out_lbl_dir, f"{prefix}{stem}.txt"))

    n_imgs = len(os.listdir(out_img_dir))
    n_lbls = len(os.listdir(out_lbl_dir))
    print(f"  Images: {n_imgs:>6} | Labels: {n_lbls:>6} → {out_img_dir}")


UDA_BASE = f"{WORKING}/uda_dataset"

print("\nBuilding UDA dataset …")
build_uda_split(RUOD_TRAIN_IMG, RUOD_TRAIN_LBL, ENH_RUOD_TRAIN,
                f"{UDA_BASE}/train/images", f"{UDA_BASE}/train/labels")
build_uda_split(RUOD_VAL_IMG,   RUOD_VAL_LBL,   ENH_RUOD_VAL,
                f"{UDA_BASE}/valid/images", f"{UDA_BASE}/valid/labels")
build_uda_split(RUOD_TEST_IMG,  RUOD_TEST_LBL,  ENH_RUOD_TEST,
                f"{UDA_BASE}/test/images",  f"{UDA_BASE}/test/labels")

print("UDA dataset ready.")

# ============================================================
# CELL 12: Detect Class Names + Write YAML
# ============================================================
def get_num_classes(label_dir: str) -> int:
    ids = set()
    for p in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(p) as f:
            for line in f:
                line = line.strip()
                if line:
                    ids.add(int(line.split()[0]))
    return max(ids) + 1 if ids else 10


def get_class_names(label_dir: str, img_dir: str, nc: int) -> list[str]:
    search_dirs = [
        label_dir,
        os.path.dirname(label_dir),
        os.path.dirname(os.path.dirname(label_dir)),
        img_dir,
        os.path.dirname(img_dir),
    ]
    for d in search_dirs:
        p = os.path.join(d, "classes.txt")
        if os.path.exists(p):
            names = [l.strip() for l in open(p) if l.strip()]
            print(f"  classes.txt found: {p}")
            return names

    for d in search_dirs:
        for yaml_f in glob.glob(os.path.join(d, "*.yaml")):
            with open(yaml_f) as f:
                data = yaml.safe_load(f)
            if "names" in data:
                names = data["names"]
                if isinstance(names, dict):
                    return [names[i] for i in sorted(names.keys())]
                return names

    print("  No class file found → using numeric names.")
    return [str(i) for i in range(nc)]


NC           = get_num_classes(RUOD_TRAIN_LBL)
CLASS_NAMES  = get_class_names(RUOD_TRAIN_LBL, RUOD_TRAIN_IMG, NC)
print(f"Classes ({NC}): {CLASS_NAMES}")

YAML_PATH = f"{WORKING}/uda_dataset.yaml"
yaml_dict = {
    "path":  UDA_BASE,
    "train": "train/images",
    "val":   "valid/images",
    "test":  "test/images",
    "nc":    len(CLASS_NAMES),
    "names": CLASS_NAMES,
}
with open(YAML_PATH, "w") as f:
    yaml.dump(yaml_dict, f, default_flow_style=False)

print(f"YAML saved → {YAML_PATH}")

# ============================================================
# CELL 13: Train YOLOv8s on UDA Dataset
# ============================================================
from ultralytics import YOLO

yolo_model = YOLO("yolov8s.pt")

results = yolo_model.train(
    data     = YAML_PATH,
    epochs   = 30,
    imgsz    = 640,
    batch    = 16,
    device   = 0 if torch.cuda.is_available() else "cpu",
    project  = f"{WORKING}/yolo_runs",
    name     = "uda_yolov8s_cbam",
    patience = 10,
    save     = True,
    exist_ok = True,
    verbose  = True,
    workers  = 2,
    # Augmentation
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    hsv_v    = 0.4,
    flipud   = 0.1,
    fliplr   = 0.5,
    mosaic   = 1.0,
    mixup    = 0.1,
)

BEST_MODEL_PATH = f"{WORKING}/yolo_runs/uda_yolov8s_cbam/weights/best.pt"
print(f"\nTraining complete. Best weights → {BEST_MODEL_PATH}")

# ============================================================
# CELL 14: Multi-Domain Evaluation (RUOD / LSUI / EUVP)
# ============================================================
def write_domain_yaml(
    test_img_dir: str,
    test_lbl_dir: str,
    class_names:  list[str],
    yaml_out:     str,
) -> str | None:
    """Create a minimal YAML pointing to a domain's test split."""
    if not (os.path.isdir(test_img_dir) and os.path.isdir(test_lbl_dir)):
        return None
    base = os.path.dirname(os.path.dirname(test_img_dir))
    d = {
        "path":  base,
        "train": "test/images",   # dummy – only test is used
        "val":   "test/images",
        "test":  "test/images",
        "nc":    len(class_names),
        "names": class_names,
    }
    with open(yaml_out, "w") as f:
        yaml.dump(d, f, default_flow_style=False)
    return yaml_out


best_model = YOLO(BEST_MODEL_PATH)
INFER_DEVICE = 0 if torch.cuda.is_available() else "cpu"

domains = {
    "RUOD": {
        "yaml":    YAML_PATH,
        "enh_dir": ENH_RUOD_TEST,
    },
    "LSUI": {
        "yaml": write_domain_yaml(
            LSUI_TEST_IMG, LSUI_TEST_LBL, CLASS_NAMES,
            f"{WORKING}/lsui_eval.yaml"
        ),
        "enh_dir": ENH_LSUI_TEST,
    },
    "EUVP": {
        "yaml": write_domain_yaml(
            EUVP_TEST_IMG, EUVP_TEST_LBL, CLASS_NAMES,
            f"{WORKING}/euvp_eval.yaml"
        ),
        "enh_dir": ENH_EUVP_TEST,
    },
}

print("\n" + "="*65)
print(f"{'DOMAIN':<10} {'mAP@0.5':>10} {'mAP@.5:.95':>12} {'Precision':>11} {'Recall':>9}")
print("="*65)

eval_results = {}
for domain, cfg in domains.items():
    if cfg["yaml"] is None:
        print(f"{domain:<10}  [SKIPPED – YAML not found]")
        continue
    
    # Folder check to prevent FileNotFoundError
    if not os.path.exists(cfg["enh_dir"]):
        print(f"{domain:<10}  [SKIPPED – Enhanced directory not found: {cfg['enh_dir']}]")
        continue

    m = best_model.val(
        data    = cfg["yaml"],
        split   = "test",
        imgsz   = 640,
        batch   = 16,
        device  = INFER_DEVICE,
        verbose = False,
    )
    eval_results[domain] = m
    
    # Safe print
    n_images = len(os.listdir(cfg['enh_dir']))
    print(f"DEBUG: Evaluating Domain {domain} on {n_images} enhanced images.")
    print(f"INFO: Detection model trained on RUOD source domain. Zero-shot testing on {domain} domain.")
    
    print(
        f"{domain:<10} "
        f"{m.box.map50:>10.4f} "
        f"{m.box.map:>12.4f} "
        f"{m.box.mp:>11.4f} "
        f"{m.box.mr:>9.4f}"
    )

# Add this log inside the domain evaluation loop
print(f"DEBUG: Evaluating Domain {domain} on {len(os.listdir(cfg['enh_dir']))} enhanced images.")
print(f"INFO: Detection model trained on RUOD source domain. Zero-shot testing on {domain} domain.")
print("="*65)

import cv2

def calculate_uciqe(img_path):
    img = cv2.imread(img_path)
    if img is None: return 0
    
    # Corrected syntax: COLOR_BGR2LAB
    img_lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(img_lab)
    
    # Sigma_c: Chromaticity variance
    sigma_c = np.sqrt(np.var(a) + np.var(b))
    # Con_l: Contrast of luminance
    con_l = np.percentile(l, 99) - np.percentile(l, 1)
    # Mu_c: Average saturation
    mu_c = np.sqrt(np.mean(a)**2 + np.mean(b)**2)
    
    # Standard UCIQE formula (Standard coefficients for underwater research)
    uciqe_score = 0.4680 * sigma_c + 0.2745 * con_l + 0.2576 * mu_c
    return uciqe_score

# ============================================================
# CELL 15: Sample Inference Visualisation (Enhanced Images)
# ============================================================
np.random.seed(42)
COLORS = {
    i: tuple(np.random.randint(80, 230, size=3).tolist())
    for i in range(len(CLASS_NAMES))
}

test_imgs = (
    glob.glob(os.path.join(ENH_RUOD_TEST, "*.jpg")) +
    glob.glob(os.path.join(ENH_RUOD_TEST, "*.png"))
)
sample_imgs = random.sample(test_imgs, min(6, len(test_imgs)))

n_cols = 3
n_rows = (len(sample_imgs) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 5 * n_rows))
axes_flat  = axes.flatten() if n_rows > 1 else list(axes)

for ax in axes_flat:
    ax.axis("off")

for idx, img_path in enumerate(sample_imgs):
    res = best_model.predict(
        source  = img_path,
        imgsz   = 640,
        conf    = 0.25,
        device  = INFER_DEVICE,
        verbose = False,
    )[0]

    img = Image.open(img_path).convert("RGB")
    ax  = axes_flat[idx]
    ax.imshow(img)

    boxes = res.boxes
    if boxes is not None and len(boxes):
        for box in boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0].item())
            conf   = float(box.conf[0].item())
            label  = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            color  = tuple(c / 255.0 for c in COLORS[cls_id % len(COLORS)])

            rect = patches.FancyBboxPatch(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color,
                facecolor=(*color, 0.08), boxstyle="round,pad=1",
            )
            ax.add_patch(rect)

            txt = ax.text(
                x1, y1 - 6, f"{label} {conf:.2f}",
                fontsize=9, color="white", fontweight="bold", va="bottom",
            )
            txt.set_path_effects([
                pe.Stroke(linewidth=3, foreground=color), pe.Normal()
            ])

    n_det = len(boxes) if boxes is not None else 0
    ax.set_title(f"{os.path.basename(img_path)} | {n_det} detections", fontsize=9)
    ax.axis("off")
    
fig.suptitle(
    "YOLOv8s-UDA — Underwater Detection (CBAM + Hybrid Loss)",
    fontsize=14, y=1.01,
)
plt.tight_layout()
save_path = f"{WORKING}/predictions_with_labels.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {save_path}")

# Evaluate Enhancement Quality (UCIQE) - Loop bayata ki jarupu
print("\n" + "="*30)
print(" ENHANCEMENT QUALITY (UCIQE)")
print("="*30)

enhanced_test_folders = [ENH_RUOD_TEST, ENH_LSUI_TEST, ENH_EUVP_TEST]
for folder in enhanced_test_folders:
    if os.path.exists(folder):
        imgs = glob.glob(os.path.join(folder, "*.jpg")) + glob.glob(os.path.join(folder, "*.png"))
        sample_imgs_uciqe = random.sample(imgs, min(50, len(imgs)))
        scores = [calculate_uciqe(p) for p in sample_imgs_uciqe]
        if scores:
            avg_score = np.mean(scores)
            domain_name = folder.split('/')[-3].upper() # Path nundi domain name teestundi
            print(f"Domain: {domain_name:<10} | Avg UCIQE: {avg_score:.4f}")
# ============================================================
# CELL 16: Per-Class Detection Summary (RUOD Test Set)
# ============================================================
all_test_imgs = (
    glob.glob(os.path.join(RUOD_TEST_IMG, "*.jpg")) +
    glob.glob(os.path.join(RUOD_TEST_IMG, "*.png"))
)

class_counter = Counter()
total_images  = 0
total_dets    = 0

for img_path in all_test_imgs:
    res = best_model.predict(
        source  = img_path,
        imgsz   = 640,
        conf    = 0.25,
        device  = INFER_DEVICE,
        verbose = False,
    )[0]
    total_images += 1
    if res.boxes is not None:
        for box in res.boxes:
            cls_id = int(box.cls[0].item())
            label  = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else str(cls_id)
            class_counter[label] += 1
            total_dets += 1

print("\n" + "="*45)
print(f"{'CLASS':<22} {'DETECTIONS':>12}")
print("="*45)
for cls_name, cnt in sorted(class_counter.items(), key=lambda x: -x[1]):
    print(f"{cls_name:<22} {cnt:>12}")
print("="*45)
print(f"{'TOTAL IMAGES':<22} {total_images:>12}")
print(f"{'TOTAL DETECTIONS':<22} {total_dets:>12}")
print("="*45)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.5 MB/s eta 0:00:0000:01
Device: cuda
Building enhancement DataLoader …
  [UIEB]   890 pairs found
  [LSUI]  4279 pairs found
  [EUVP]  3700 pairs found
  Combined:  8869 pairs | batches: 555
Enhancer parameters: 508,849
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:03<00:00, 167MB/s]  



 Training LightUNet-CBAM with Hybrid Loss
Epoch [ 1/10] Loss: 0.5169 | L1: 0.0899 | SSIM: 0.2588 | Perc: 2.9642 | Color: 0.0224 | LR: 9.76e-04
Epoch [ 2/10] Loss: 0.4573 | L1: 0.0746 | SSIM: 0.2098 | Perc: 2.7668 | Color: 0.0243 | LR: 9.05e-04
Epoch [ 3/10] Loss: 0.4442 | L1: 0.0710 | SSIM: 0.2006 | Perc: 2.7164 | Color: 0.0243 | LR: 7.96e-04
Epoch [ 4/10] Loss: 0.4367 | L1: 0.0689 | SSIM: 0.1955 | Perc: 2.6877 | Color: 0.0249 | LR: 6.58e-04
Epoch [ 5/10] Loss: 0.4303 | L1: 0.0672 | SSIM: 0.1913 | Perc: 2.6617 | Color: 0.0259 | LR: 5.05e-04
Epoch [ 6/10] Loss: 0.4245 | L1: 0.0653 | SSIM: 0.1874 | Perc: 2.6419 | Color: 0.0260 | LR: 3.52e-04
Epoch [ 7/10] Loss: 0.4191 | L1: 0.0637 | SSIM: 0.1838 | Perc: 2.6216 | Color: 0.0266 | LR: 2.14e-04
Epoch [ 8/10] Loss: 0.4157 | L1: 0.0625 | SSIM: 0.1815 | Perc: 2.6108 | Color: 0.0270 | LR: 1.05e-04
Epoch [ 9/10] Loss: 0.4128 | L1: 0.0614 | SSIM: 0.1797 | Perc: 2.6017 | Color: 0.0272 | LR: 3.42e-05
Epoch [10/10] Loss: 0.4110 | L1: 0.0608 | SSIM: 

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/enhanced/euvp/test/images'